# Enterprise Migration Copilot — Fine-tuning

Fine-tunes a small code model on the enterprise migration dataset using QLoRA.

**Run this notebook 3 times, changing `MODEL_NAME` each session:**
- Session 1: `deepseek-ai/deepseek-coder-1.3b-instruct`
- Session 2: `Qwen/Qwen2.5-Coder-1.5B-Instruct`
- Session 3: `microsoft/Phi-3.5-mini-instruct`

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Add `HF_TOKEN` to Colab Secrets (🔑 key icon in left sidebar)
3. Set `MODEL_NAME` in Cell 2 for this session

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers==4.44.0 datasets peft==0.12.0 trl==0.10.1 bitsandbytes==0.43.3 accelerate==0.33.0 huggingface_hub

In [ ]:
# Cell 2 — Configuration
# ====================================================
# CHANGE MODEL_NAME FOR EACH SESSION:
# Session 1: "deepseek-ai/deepseek-coder-1.3b-instruct"
# Session 2: "Qwen/Qwen2.5-Coder-1.5B-Instruct"
# Session 3: "microsoft/Phi-3.5-mini-instruct"
# ====================================================
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

HF_USERNAME = "praveends"
DATASET_REPO = f"{HF_USERNAME}/enterprise-migration-dataset"

# Auto-generate output repo name from model name
model_short = MODEL_NAME.split('/')[-1].lower().replace('.', '-')
OUTPUT_REPO = f"{HF_USERNAME}/migration-copilot-{model_short}"

print(f"Model:   {MODEL_NAME}")
print(f"Dataset: {DATASET_REPO}")
print(f"Output:  {OUTPUT_REPO}")

In [ ]:
# Cell 3 — HuggingFace login
# Token stored in Colab Secrets — never appears in notebook
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in to HuggingFace")

In [ ]:
# Cell 4 — Load and format dataset
from datasets import load_dataset

dataset = load_dataset(DATASET_REPO, data_files="train.jsonl", split="train")
print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

def format_prompt(example):
    """
    Format each training pair as an instruction-following prompt.
    This exact format is used at inference time by the migrator agent.
    """
    source_lang = example.get('source_language', 'sql').upper()
    difficulty = example.get('difficulty', 'medium')
    source_code = example.get('source_code', '')
    pyspark_code = example.get('pyspark_code', '')

    prompt = f"""### Instruction:
Convert the following {source_lang} code to PySpark.
Difficulty: {difficulty}

### Input:
{source_code}

### Response:
{pyspark_code}"""

    return {"text": prompt}

dataset = dataset.map(format_prompt)

# 95/5 train/eval split
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print("\nSample prompt (first 500 chars):")
print(train_dataset[0]["text"][:500])

In [ ]:
# Cell 5 — Load model with 4-bit quantization (QLoRA)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

print(f"\nModel loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Cell 6 — Apply LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1-2% of total params — that's normal for LoRA

In [ ]:
# Cell 7 — Fine-tune with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"/tmp/{model_short}-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
    load_best_model_at_end=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=1024,
)

print(f"Training on {len(train_dataset)} examples for {training_args.num_train_epochs} epochs")
print("Starting training...")
trainer.train()
print("\nTraining complete!")

In [ ]:
# Cell 8 — Push fine-tuned model to HuggingFace
print(f"Pushing model to: {OUTPUT_REPO}")
trainer.model.push_to_hub(OUTPUT_REPO)
tokenizer.push_to_hub(OUTPUT_REPO)
print(f"\nDone! Model live at:")
print(f"https://huggingface.co/{OUTPUT_REPO}")

In [ ]:
# Cell 9 — Quick inference test to verify the fine-tuned model works
test_prompt = """### Instruction:
Convert the following SQL code to PySpark.
Difficulty: medium

### Input:
SELECT customer_id, SUM(amount) AS total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY total DESC;

### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
generated = response[len(test_prompt):].strip()

print("=" * 60)
print("FINE-TUNED MODEL OUTPUT:")
print("=" * 60)
print(generated)
print("=" * 60)
print("\nIf the output above looks like valid PySpark, training succeeded.")

## After This Session

1. Note the final **train loss** and **eval loss** from Cell 7 output
2. Save the inference test output from Cell 9
3. Verify the model is live at `https://huggingface.co/{OUTPUT_REPO}`
4. Start a new Colab session for the next model

**Session checklist:**
- [ ] Session 1: `deepseek-ai/deepseek-coder-1.3b-instruct` → `migration-copilot-deepseek-coder-1-3b-instruct`
- [ ] Session 2: `Qwen/Qwen2.5-Coder-1.5B-Instruct` → `migration-copilot-qwen2-5-coder-1-5b-instruct`
- [ ] Session 3: `microsoft/Phi-3.5-mini-instruct` → `migration-copilot-phi-3-5-mini-instruct`